# EP1 — Pipeline RAG para AsesorBot — FinanChile Asesorías S.A.
## Evaluación Parcial N°1 — ISY0101 Ingeniería de Soluciones con IA
**Alumno:** José Muñoz

En este notebook construimos el pipeline RAG completo para AsesorBot. La idea es que el asistente pueda responder preguntas de clientes usando los documentos reales de FinanChile como fuente de información.

#### Cómo funciona el pipeline:
```
DOCUMENTOS INTERNOS          FUENTES EXTERNAS
(Manual, Políticas, FAQ)     (Regulaciones CMF, Indicadores BC)
        │                           │
        └──────────┬────────────────┘
                   ▼
          CARGA Y CHUNKING
                   │
                   ▼
          EMBEDDINGS (text-embedding-3-small)
                   │
                   ▼
          VECTOR STORE (FAISS)
                   │
    ┌─────────────┤
    │              ▼
QUERY USUARIO  RECUPERACIÓN (Top-K)
    │              │
    └────────────┤
                   ▼
          GENERACIÓN (GPT-4o)
                   │
                   ▼
          RESPUESTA con fuente citada
```

## 1. Instalación de Dependencias

In [1]:
!pip install openai langchain langchain-openai langchain-community faiss-cpu python-dotenv requests

# Cargar variables de entorno desde .env
from dotenv import load_dotenv
load_dotenv()
print('Variables de entorno cargadas.')

Variables de entorno cargadas.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Configuración del Entorno

In [5]:
import os
from pathlib import Path
from datetime import datetime

# LangChain imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Configuración del entorno para usar GitHub Models
os.environ["OPENAI_API_KEY"] = os.getenv("GITHUB_TOKEN", "")
os.environ["OPENAI_API_BASE"] = os.getenv("GITHUB_BASE_URL", "https://models.inference.ai.azure.com")

# Inicialización de modelos
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_base=os.environ["OPENAI_API_BASE"]
)

llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.1,
    openai_api_base=os.environ["OPENAI_API_BASE"]
)

print("✅ Entorno configurado.")
print(f"   LLM: gpt-4o | Temperatura: 0.1")
print(f"   Embeddings: text-embedding-3-small")
print(f"   Vector Store: FAISS (in-memory)")

✅ Entorno configurado.
   LLM: gpt-4o | Temperatura: 0.1
   Embeddings: text-embedding-3-small
   Vector Store: FAISS (in-memory)


## 3. Carga de Fuentes Internas

Las **fuentes internas** son los documentos propios de FinanChile: el manual de productos, las políticas de la empresa y el FAQ. Son el núcleo del sistema porque tienen toda la información específica que el LLM no conoce por sí solo.

In [6]:
# =====================================================================
# PASO 1: CARGA DE FUENTES INTERNAS
# =====================================================================

DATA_DIR = Path("./data")

# Definición de fuentes internas con metadata
INTERNAL_SOURCES = [
    {
        "path": DATA_DIR / "manual_productos_financieros.txt",
        "source_name": "Manual de Productos Financieros",
        "source_type": "interno",
        "priority": "alta"
    },
    {
        "path": DATA_DIR / "politicas_empresa.txt",
        "source_name": "Políticas y Procedimientos Internos",
        "source_type": "interno",
        "priority": "alta"
    },
    {
        "path": DATA_DIR / "faq_financhile.txt",
        "source_name": "FAQ FinanChile",
        "source_type": "interno",
        "priority": "media"
    }
]

def load_internal_documents(sources: list) -> list[Document]:
    """Carga documentos internos y los convierte a objetos Document de LangChain con metadata."""
    documents = []
    for source in sources:
        try:
            content = source["path"].read_text(encoding="utf-8")
            doc = Document(
                page_content=content,
                metadata={
                    "source": source["source_name"],
                    "source_type": source["source_type"],
                    "priority": source["priority"],
                    "file": str(source["path"]),
                    "loaded_at": datetime.now().isoformat()
                }
            )
            documents.append(doc)
            print(f"  ✅ Cargado: {source['source_name']} ({len(content)} caracteres)")
        except FileNotFoundError:
            print(f"  ⚠️  Archivo no encontrado: {source['path']}")
    return documents

print("📂 Cargando fuentes internas...")
internal_docs = load_internal_documents(INTERNAL_SOURCES)
print(f"\n📊 Total fuentes internas: {len(internal_docs)} documentos")

📂 Cargando fuentes internas...
  ✅ Cargado: Manual de Productos Financieros (7759 caracteres)
  ✅ Cargado: Políticas y Procedimientos Internos (5965 caracteres)
  ✅ Cargado: FAQ FinanChile (4832 caracteres)

📊 Total fuentes internas: 3 documentos


## 4. Carga de Fuentes Externas

Las **fuentes externas** son las regulaciones de la CMF y los indicadores económicos del Banco Central. Las incluimos para que el sistema pueda responder preguntas sobre el contexto regulatorio — por ejemplo, si una tasa ofrecida cumple con los límites legales vigentes.

In [7]:
# =====================================================================
# PASO 2: CARGA DE FUENTES EXTERNAS
# =====================================================================

def load_external_documents(sources: list) -> list[Document]:
    """Carga el documento externo de regulaciones CMF (archivo local que simula la fuente externa)."""
    documents = []
    for source in sources:
        try:
            content = source["path"].read_text(encoding="utf-8")
            doc = Document(
                page_content=content,
                metadata={
                    "source": source["source_name"],
                    "source_type": "externo",
                    "url": source.get("url", ""),
                    "loaded_at": datetime.now().isoformat()
                }
            )
            documents.append(doc)
            print(f"  ✅ Cargado: {source['source_name']} ({len(content)} caracteres)")
        except Exception as e:
            print(f"  ⚠️  Error cargando {source.get('source_name', 'desconocido')}: {e}")
    return documents

def get_banco_central_indicators() -> Document:
    """Obtiene indicadores económicos del Banco Central (simulado con valores actuales)."""
    # En producción, se conectaría a: https://si3.bcentral.cl/siete/ES/Siete/Cuadro/CAP_IND_MACRO
    # Para el proyecto, usamos valores de referencia de enero 2025 documentados en regulaciones_cmf.txt
    indicadores_texto = """INDICADORES ECONÓMICOS BANCO CENTRAL DE CHILE — Enero 2025
    Fuente: Banco Central de Chile (www.bcentral.cl)
    
    Valor UF (17/01/2025): $38.453,21 CLP
    Valor UTM (Enero 2025): $67.294 CLP
    Tasa de Política Monetaria (TPM): 5,0% anual
    IPC últimos 12 meses (Diciembre 2024): 4,5% anual
    Tipo de cambio USD/CLP (promedio enero): $979 CLP por USD
    Tasa captación bancaria 30-89 días: 4,8% anual
    Tasa captación bancaria 90-365 días: 5,1% anual
    Tasa colocación consumo promedio mercado: 23,5% anual
    Tasa colocación hipotecario promedio mercado: 4,8% anual (en UF)"""
    
    return Document(
        page_content=indicadores_texto,
        metadata={
            "source": "Banco Central de Chile — Indicadores Económicos",
            "source_type": "externo",
            "url": "https://www.bcentral.cl",
            "loaded_at": datetime.now().isoformat()
        }
    )

EXTERNAL_SOURCES = [
    {
        "path": DATA_DIR / "regulaciones_cmf.txt",
        "source_name": "Normativa CMF — Comisión para el Mercado Financiero",
        "url": "https://www.cmfchile.cl"
    }
]

print("🌐 Cargando fuentes externas...")
external_docs = load_external_documents(EXTERNAL_SOURCES)

# Agregar indicadores del Banco Central
bc_doc = get_banco_central_indicators()
external_docs.append(bc_doc)
print(f"  ✅ Cargado: {bc_doc.metadata['source']} (indicadores en tiempo real simulados)")

print(f"\n📊 Total fuentes externas: {len(external_docs)} documentos")

# Resumen total
all_docs = internal_docs + external_docs
print(f"\n📚 TOTAL DOCUMENTOS EN KNOWLEDGE BASE: {len(all_docs)}")
print("   - Internos:", sum(1 for d in all_docs if d.metadata.get('source_type') == 'interno'))
print("   - Externos:", sum(1 for d in all_docs if d.metadata.get('source_type') == 'externo'))

🌐 Cargando fuentes externas...
  ✅ Cargado: Normativa CMF — Comisión para el Mercado Financiero (5439 caracteres)
  ✅ Cargado: Banco Central de Chile — Indicadores Económicos (indicadores en tiempo real simulados)

📊 Total fuentes externas: 2 documentos

📚 TOTAL DOCUMENTOS EN KNOWLEDGE BASE: 5
   - Internos: 3
   - Externos: 2


## 5. Chunking (Fragmentación de Documentos)

Dividimos los documentos en fragmentos pequeños (chunks) para que el retriever pueda encontrar exactamente la sección relevante a cada pregunta, en lugar de pasarle todo el documento al modelo.

**Por qué chunk_size=500:** Los documentos financieros tienen secciones bien definidas (un producto, una política). Con 500 caracteres capturamos generalmente una sección completa sin mezclar información de distintos productos.

**Por qué chunk_overlap=100 (20%):** Las condiciones de un producto a veces quedan cortadas en el límite entre dos chunks. El solapamiento del 20% evita ese problema.

In [8]:
# =====================================================================
# PASO 3: CHUNKING
# =====================================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]  # Orden de separadores preferidos
)

print("✂️  Aplicando chunking a todos los documentos...")
print(f"   chunk_size: 500 | chunk_overlap: 100")

chunks = text_splitter.split_documents(all_docs)

print(f"\n📊 RESULTADOS DEL CHUNKING:")
print(f"   Documentos originales: {len(all_docs)}")
print(f"   Chunks generados: {len(chunks)}")
print(f"   Promedio de chars por chunk: {sum(len(c.page_content) for c in chunks) // len(chunks)}")

# Mostrar distribución por fuente
print("\n📋 Distribución por fuente:")
from collections import Counter
source_counts = Counter(c.metadata.get('source', 'desconocido') for c in chunks)
for source, count in source_counts.most_common():
    print(f"   {source[:55]:55s}: {count:3d} chunks")

# Mostrar ejemplo de chunk
print("\n🔍 Ejemplo de chunk generado:")
print(f"   Fuente: {chunks[0].metadata.get('source')}")
print(f"   Tipo: {chunks[0].metadata.get('source_type')}")
print(f"   Contenido: {chunks[0].page_content[:200]}...")

✂️  Aplicando chunking a todos los documentos...
   chunk_size: 500 | chunk_overlap: 100

📊 RESULTADOS DEL CHUNKING:
   Documentos originales: 5
   Chunks generados: 75
   Promedio de chars por chunk: 338

📋 Distribución por fuente:
   Manual de Productos Financieros                        :  25 chunks
   Políticas y Procedimientos Internos                    :  18 chunks
   Normativa CMF — Comisión para el Mercado Financiero    :  16 chunks
   FAQ FinanChile                                         :  14 chunks
   Banco Central de Chile — Indicadores Económicos        :   2 chunks

🔍 Ejemplo de chunk generado:
   Fuente: Manual de Productos Financieros
   Tipo: interno
   Contenido: MANUAL DE PRODUCTOS FINANCIEROS — FinanChile Asesorías S.A.
Versión 3.2 | Actualizado: Enero 2025 | Uso Interno y Clientes

ADVERTENCIA...


## 6. Embeddings y Vector Store (FAISS)

Convertimos cada chunk en un vector numérico que representa su significado semántico. Después FAISS indexa esos vectores para poder buscar rápidamente cuáles son los más parecidos a la pregunta del usuario.

In [9]:
# =====================================================================
# PASO 4: EMBEDDINGS Y VECTOR STORE (FAISS)
# =====================================================================

print("🔢 Generando embeddings y construyendo vector store...")
print("   (Este proceso puede tomar unos segundos según el número de chunks)")

# Crear vector store FAISS con embeddings
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings_model
)

print(f"\n✅ Vector Store construido exitosamente.")
print(f"   Chunks indexados: {len(chunks)}")
print(f"   Modelo de embeddings: text-embedding-3-small")
print(f"   Dimensión de vectores: 1536")
print(f"   Tipo de índice: FAISS FlatL2")

# Guardar el índice localmente para reutilización
vector_store.save_local("./financhile_vectorstore")
print(f"\n💾 Vector store guardado en: ./financhile_vectorstore")

🔢 Generando embeddings y construyendo vector store...
   (Este proceso puede tomar unos segundos según el número de chunks)

✅ Vector Store construido exitosamente.
   Chunks indexados: 75
   Modelo de embeddings: text-embedding-3-small
   Dimensión de vectores: 1536
   Tipo de índice: FAISS FlatL2

💾 Vector store guardado en: ./financhile_vectorstore


## 7. Ensamblado del Pipeline RAG

Aquí conectamos todas las piezas: el retriever, el prompt maestro y el LLM en una cadena `RetrievalQA` de LangChain.

In [10]:
# =====================================================================
# PASO 5: ENSAMBLADO DEL PIPELINE RAG COMPLETO
# =====================================================================

# Configuración del retriever (top-4 chunks más relevantes)
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}  # Recupera los 4 chunks más similares semánticamente
)

# Prompt template para RAG — integra contexto recuperado
RAG_PROMPT_TEMPLATE = """Eres AsesorBot, el asistente virtual oficial de FinanChile Asesorías S.A.,
empresa regulada por la CMF de Chile (registro N°1247).

INSTRUCCIONES:
1. Responde ÚNICAMENTE basándote en el CONTEXTO proporcionado abajo.
2. Si el contexto no contiene información suficiente, di: 
   'Para esta consulta te recomiendo contactar a un asesor en el 600-FINANS.'
3. Cita la fuente de la información entre corchetes al final de tu respuesta.
4. Para consultas sobre inversiones, incluye SIEMPRE esta advertencia:
   'ADVERTENCIA CMF: Las inversiones conllevan riesgo de pérdida de capital.'
5. Responde en español chileno, con lenguaje claro y profesional.
6. Máximo 300 palabras en tu respuesta.

CONTEXTO RECUPERADO:
{context}

PREGUNTA DEL CLIENTE:
{question}

RESPUESTA FUNDAMENTADA (con fuente citada):"""

rag_prompt = PromptTemplate(
    template=RAG_PROMPT_TEMPLATE,
    input_variables=["context", "question"]
)

def format_docs(docs):
    """Concatena el contenido de los docs recuperados."""
    return "\n\n".join(doc.page_content for doc in docs)

# Pipeline LCEL: recuperar contexto → formatear → prompt → LLM → parsear
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("✅ Pipeline RAG ensamblado correctamente.")
print("   Arquitectura: LangChain LCEL (Expression Language)")
print("   Retriever: FAISS similarity search (k=4)")
print("   LLM: gpt-4o (temperatura=0.1)")
print("   Prompt: RAG personalizado con restricciones CMF")

✅ Pipeline RAG ensamblado correctamente.
   Arquitectura: LangChain LCEL (Expression Language)
   Retriever: FAISS similarity search (k=4)
   LLM: gpt-4o (temperatura=0.1)
   Prompt: RAG personalizado con restricciones CMF


## 8. Pruebas con Consultas Reales

In [11]:
# =====================================================================
# PASO 6: FUNCIÓN DE CONSULTA CON TRAZABILIDAD
# =====================================================================

def consultar_financhile(pregunta: str, verbose: bool = True) -> dict:
    """Realiza una consulta al sistema RAG de FinanChile con trazabilidad completa."""
    print(f"\n{'='*65}")
    print(f"CONSULTA: {pregunta}")
    print(f"{'='*65}")
    
    # Recuperar fuentes (para trazabilidad)
    fuentes = retriever.invoke(pregunta)
    
    # Ejecutar el pipeline RAG
    respuesta = rag_chain.invoke(pregunta)
    
    print(f"\nRESPUESTA FINANCHILE:\n")
    print(respuesta)
    
    if verbose:
        print(f"\nFUENTES UTILIZADAS ({len(fuentes)} chunks recuperados):")
        for i, doc in enumerate(fuentes, 1):
            source = doc.metadata.get('source', 'desconocido')
            source_type = doc.metadata.get('source_type', 'desconocido')
            icon = "Interno" if source_type == "interno" else "Externo"
            print(f"   [{i}] [{icon}] {source}")
            print(f"       Fragmento: {doc.page_content[:100].strip()}...")
    
    return {
        "pregunta": pregunta,
        "respuesta": respuesta,
        "fuentes": [(d.metadata.get('source'), d.metadata.get('source_type')) for d in fuentes]
    }

# =====================================================================
# PRUEBA 1: Consulta sobre producto interno
# =====================================================================
r1 = consultar_financhile("¿Cuáles son los requisitos para pedir un crédito de consumo?")


CONSULTA: ¿Cuáles son los requisitos para pedir un crédito de consumo?

RESPUESTA FINANCHILE:

Para solicitar un crédito de consumo en FinanChile Asesorías S.A., necesitas cumplir con los siguientes requisitos:

1. **Edad mínima**: Ser mayor de 21 años.
2. **Renta líquida mínima**: $400.000 CLP.
3. **Antigüedad laboral mínima**: 6 meses.
4. **Historial financiero**: No tener morosidades en el sistema financiero.
5. **Documentación requerida**:
   - Cédula de identidad vigente.
   - 3 últimas liquidaciones de sueldo (o la última declaración de renta si eres trabajador independiente).
   - Extracto bancario de los últimos 6 meses.

Estos requisitos son necesarios para evaluar tu capacidad de pago y garantizar un proceso de aprobación ágil y transparente. Recuerda que, una vez entregados todos los documentos completos, la respuesta sobre la aprobación del crédito se entrega en un plazo de 24 horas hábiles.

[Fuente: Información proporcionada en el contexto]

FUENTES UTILIZADAS (4 chunks 

In [12]:
# =====================================================================
# PRUEBA 2: Consulta que combina datos internos + externos (regulatoria)
# =====================================================================
r2 = consultar_financhile(
    "La tasa del crédito de consumo que ofrecen, ¿está dentro de los límites legales de la CMF?"
)


CONSULTA: La tasa del crédito de consumo que ofrecen, ¿está dentro de los límites legales de la CMF?

RESPUESTA FINANCHILE:

Sí, la tasa del crédito de consumo que se ofrece está dentro de los límites legales establecidos por la Comisión para el Mercado Financiero (CMF). Según el contexto proporcionado, la tasa de interés corriente para créditos de consumo personales es de 23,9% anual, mientras que la Tasa Máxima Convencional (TMC) vigente para créditos de consumo entre 200 y 5.000 UF es de 35,5% anual. Por lo tanto, la tasa ofrecida cumple con la normativa y no excede el límite permitido por la ley.

Es importante destacar que las instituciones financieras no pueden superar las TMC bajo ninguna circunstancia, ya que hacerlo constituye un delito de usura según el artículo 472 del Código Penal [Sección 3: Tasas Máximas Convencionales (TMC) — Enero 2025].

ADVERTENCIA CMF: Las inversiones conllevan riesgo de pérdida de capital.

FUENTES UTILIZADAS (4 chunks recuperados):
   [1] [Externo

In [13]:
# =====================================================================
# PRUEBA 3: Consulta sobre política de reclamos
# =====================================================================
r3 = consultar_financhile(
    "Quiero presentar un reclamo porque me cobraron una comisión que no me habían informado. ¿Cómo lo hago?"
)


CONSULTA: Quiero presentar un reclamo porque me cobraron una comisión que no me habían informado. ¿Cómo lo hago?

RESPUESTA FINANCHILE:

Para presentar un reclamo por un cobro de comisión no informado, puedes hacerlo a través de las siguientes opciones:

1. **App FinanChile**: Ingresa a la sección "Ayuda > Enviar Reclamo".  
2. **Correo electrónico**: Envía los detalles de tu reclamo a **reclamos@financhile.cl**.  
3. **Teléfono**: Llama al **600-FINANS**, selecciona la opción 3 y expón tu caso.  
4. **Presencialmente**: Acércate a cualquier sucursal de FinanChile y registra tu reclamo en el libro de reclamos.  
5. **Sitio web**: Completa el formulario en **www.financhile.cl/reclamos**.  

Tu caso corresponde a un reclamo de **Nivel 2**, ya que se trata de un cobro incorrecto. Esto significa que será revisado por el Supervisor de Área, quien debe entregarte una respuesta en un plazo máximo de **10 días hábiles**.  

Si no estás conforme con la resolución, puedes recurrir a instancias 

In [14]:
# =====================================================================
# PRUEBA 4: Consulta sobre inversión (debe incluir advertencia CMF)
# =====================================================================
r4 = consultar_financhile(
    "Tengo $500.000 pesos. ¿Me conviene el Fondo Conservador o el Crecimiento?"
)


CONSULTA: Tengo $500.000 pesos. ¿Me conviene el Fondo Conservador o el Crecimiento?

RESPUESTA FINANCHILE:

La elección entre el Fondo Conservador y el Fondo Crecimiento depende de tu horizonte de inversión y tu tolerancia al riesgo. 

Si planeas invertir por un período corto, menor a 1 año, y prefieres minimizar las fluctuaciones en el valor de tu inversión, el Fondo Conservador sería más adecuado. Este fondo invierte principalmente en renta fija, tiene menor volatilidad y una rentabilidad histórica de 6,8% anual. Además, permite rescatar tu dinero en T+2 días hábiles, aunque si lo haces antes de 30 días, se aplicará una comisión de rescate anticipado.

Por otro lado, si tu horizonte de inversión es mayor a 3 años y estás dispuesto a asumir mayor riesgo para buscar un retorno potencialmente más alto, el Fondo Crecimiento podría ser una mejor opción. Este fondo invierte 60% en acciones y tiene una rentabilidad histórica de 12,4% anual (no garantizada). Sin embargo, debes considerar qu

In [15]:
# =====================================================================
# PRUEBA 5: Consulta fuera del conocimiento del sistema (escalada)
# =====================================================================
r5 = consultar_financhile(
    "¿Pueden ayudarme con una inversión en criptomonedas?"
)


CONSULTA: ¿Pueden ayudarme con una inversión en criptomonedas?

RESPUESTA FINANCHILE:

Para esta consulta te recomiendo contactar a un asesor en el 600-FINANS. [Fuente: Base de Conocimiento para AsesorBot v2.0 | Actualizado: Enero 2025]

FUENTES UTILIZADAS (4 chunks recuperados):
   [1] [Interno] FAQ FinanChile
       Fragmento: A: El Fondo Conservador invierte principalmente en renta fija (bonos, depósitos) con rentabilidad hi...
   [2] [Interno] Manual de Productos Financieros
       Fragmento: ================================================================
SECCIÓN 1: PRODUCTOS DE AHORRO E IN...
   [3] [Interno] FAQ FinanChile
       Fragmento: PREGUNTAS FRECUENTES (FAQ) — FinanChile Asesorías S.A.
Base de Conocimiento para AsesorBot v2.0 | Ac...
   [4] [Interno] Manual de Productos Financieros
       Fragmento: - Comisión de administración: 1,2% anual sobre el patrimonio.
- Comisión de ingreso: 0% (sin costo d...


## 9. Evaluación de Coherencia (IE4)

Verificamos que las respuestas generadas sean coherentes con los documentos fuente. Para eso chequeamos si la respuesta incluye las palabras clave que debería mencionar según la pregunta.

In [17]:
# =====================================================================
# ANÁLISIS DE COHERENCIA (IE4)
# =====================================================================

def evaluar_coherencia(pregunta: str, respuesta_esperada_keywords: list) -> dict:
    """Evalúa si la respuesta del RAG contiene los conceptos clave esperados."""
    fuentes = retriever.invoke(pregunta)
    respuesta = rag_chain.invoke(pregunta).lower()
    
    # Verificar presencia de keywords esperadas
    keywords_presentes = [kw for kw in respuesta_esperada_keywords if kw.lower() in respuesta]
    keywords_ausentes = [kw for kw in respuesta_esperada_keywords if kw.lower() not in respuesta]
    cobertura = len(keywords_presentes) / len(respuesta_esperada_keywords) * 100
    
    # Verificar uso de fuentes relevantes
    tipos_fuente = set(d.metadata.get('source_type') for d in fuentes)
    
    return {
        "pregunta": pregunta,
        "cobertura_conceptual": f"{cobertura:.0f}%",
        "keywords_presentes": keywords_presentes,
        "keywords_ausentes": keywords_ausentes,
        "tipos_fuente_usados": list(tipos_fuente),
        "num_fuentes": len(fuentes)
    }

# Casos de prueba para evaluar coherencia
casos_coherencia = [
    {
        "pregunta": "¿Cuánto es la tasa del depósito a plazo a 90 días?",
        "keywords_esperadas": ["5,8%", "90", "anual"]
    },
    {
        "pregunta": "¿Cuál es el monto mínimo para el Fondo Conservador?",
        "keywords_esperadas": ["100.000", "conservador"]
    },
    {
        "pregunta": "¿En cuántos días hábiles deben responder mi reclamo?",
        "keywords_esperadas": ["10", "días hábiles", "reclamo"]
    },
    {
        "pregunta": "¿Cuál es la tasa máxima de crédito de consumo según la CMF?",
        "keywords_esperadas": ["51", "cmf", "tasa máxima"]
    }
]

print("EVALUACIÓN DE COHERENCIA RAG — AsesorBot")
print("=" * 65)

resultados_coherencia = []
for caso in casos_coherencia:
    eval_result = evaluar_coherencia(caso["pregunta"], caso["keywords_esperadas"])
    resultados_coherencia.append(eval_result)
    
    print(f"\nPregunta: {eval_result['pregunta']}")
    print(f"   Cobertura conceptual: {eval_result['cobertura_conceptual']}")
    print(f"   Keywords encontradas: {eval_result['keywords_presentes']}")
    if eval_result['keywords_ausentes']:
        print(f"   Keywords faltantes: {eval_result['keywords_ausentes']}")
    print(f"   Fuentes usadas: {eval_result['tipos_fuente_usados']} ({eval_result['num_fuentes']} chunks)")

cobertura_promedio = sum(
    float(r['cobertura_conceptual'].replace('%','')) for r in resultados_coherencia
) / len(resultados_coherencia)

print(f"\n{'='*65}")
print(f"COBERTURA CONCEPTUAL PROMEDIO: {cobertura_promedio:.1f}%")
print(f"   (Meta: >80% para considerar el sistema coherente)")
if cobertura_promedio >= 80:
    print("   RESULTADO: Sistema coherente. El RAG responde con los conceptos esperados.")
else:
    print("   RESULTADO: Coherencia insuficiente. Revisar chunking o fuentes.")

EVALUACIÓN DE COHERENCIA RAG — AsesorBot

Pregunta: ¿Cuánto es la tasa del depósito a plazo a 90 días?
   Cobertura conceptual: 100%
   Keywords encontradas: ['5,8%', '90', 'anual']
   Fuentes usadas: ['externo', 'interno'] (4 chunks)

Pregunta: ¿Cuál es el monto mínimo para el Fondo Conservador?
   Cobertura conceptual: 100%
   Keywords encontradas: ['100.000', 'conservador']
   Fuentes usadas: ['interno'] (4 chunks)

Pregunta: ¿En cuántos días hábiles deben responder mi reclamo?
   Cobertura conceptual: 100%
   Keywords encontradas: ['10', 'días hábiles', 'reclamo']
   Fuentes usadas: ['interno'] (4 chunks)

Pregunta: ¿Cuál es la tasa máxima de crédito de consumo según la CMF?
   Cobertura conceptual: 100%
   Keywords encontradas: ['51', 'cmf', 'tasa máxima']
   Fuentes usadas: ['externo', 'interno'] (4 chunks)

COBERTURA CONCEPTUAL PROMEDIO: 100.0%
   (Meta: >80% para considerar el sistema coherente)
   RESULTADO: Sistema coherente. El RAG responde con los conceptos esperados.


## 10. Resumen del Pipeline

El pipeline que construimos cubre todos los requerimientos del caso:

| Componente | Implementación | Por qué lo elegimos |
|------------|---------------|---------------------|
| **Fuentes internas** | Manual, Políticas, FAQ | Son el conocimiento específico de FinanChile |
| **Fuentes externas** | Normativa CMF, Banco Central | Contexto regulatorio actualizado |
| **Chunking** | 500 chars / 100 overlap | Balance entre granularidad y contexto |
| **Embeddings** | text-embedding-3-small | Buena calidad a bajo costo |
| **Vector Store** | FAISS | Fácil de usar, sin costo operacional |
| **Retriever** | Similarity, k=4 | Suficiente contexto sin meter ruido |
| **LLM** | GPT-4o, T=0.1 | Precisión factual en dominio regulado |
| **Trazabilidad** | Fuente citada en cada respuesta | Lo exige la CMF para auditoría |